# MaxEnt species-distribution modelling

Reproduces the manuscript's biodiversity workflow for the five IUCN species.
Replace the synthetic occurrences below with the GBIF download produced by
`scripts/download_datasets.py --gbif`.

In [ ]:
import numpy as np, pandas as pd
from cmhr.species_distribution import MaxEntModel, EnsembleSDM, screen_multicollinearity
rng = np.random.default_rng(0)
presences = pd.DataFrame({'bio1':rng.normal(24,1.5,80),'bio12':rng.normal(1700,200,80),'elev':rng.normal(700,80,80),'ndvi':rng.normal(0.7,0.05,80)})
background = pd.DataFrame({'bio1':rng.normal(25,2,500),'bio12':rng.normal(1500,300,500),'elev':rng.normal(800,200,500),'ndvi':rng.normal(0.55,0.1,500)})
kept = screen_multicollinearity(presences, max_pearson=0.8); kept

In [ ]:
ens = EnsembleSDM(replicates=10).fit(presences[kept], background[kept])
X_grid = pd.DataFrame({'bio1':np.linspace(20,28,200),'bio12':np.linspace(1300,2100,200),
                       'elev':np.linspace(500,900,200),'ndvi':np.linspace(0.4,0.85,200)})
pred = ens.predict(X_grid[kept])
print('mean AUC:', ens.aucs[:5], 'mean TSS:', ens.tsses[:5])
pd.DataFrame({'mean':pred.mean_suitability,'std':pred.std_suitability}).head()